# LLM Roundtable — Judge Run Analysis

Lexical comparison of model outputs across the pipeline: primary responses, peer critiques,
refined responses, and the judge-selected response for each author.

**Metrics** — ROUGE-1, ROUGE-2, ROUGE-L (F-measure) and TF-IDF cosine similarity; lengths in words.

## Setup and Metric Definitions

In [1]:
from __future__ import annotations

import json
import re
from itertools import combinations
from pathlib import Path

import pandas as pd
from rouge_score import rouge_scorer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

RUN_PATH = None          # None -> newest outputs/n8n/*/judge_run.json
OUTPUT_ROOT = Path("outputs/n8n")
SAVE_CSV = True

run_path = Path(RUN_PATH) if RUN_PATH else max(
    OUTPUT_ROOT.glob("*/judge_run.json"), key=lambda p: p.stat().st_mtime)
run = json.loads(run_path.read_text(encoding="utf-8"))

models = run["models"]
primary = run["primary_responses"]
critiques = run["critiques"]      # critiques[i][j] = critique by j about i's primary
refined = run["refined_responses"]  # refined[i][j]  = i's response after j's critique
judgments = run["judgments"]
n = len(models)
short = [m.split("/")[-1] for m in models]

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)


def words(t):
    return len((t or "").split())


def cosine(a, b):
    a, b = a or "", b or ""
    if not a.strip() or not b.strip():
        return 0.0
    vec = TfidfVectorizer(max_features=4096, stop_words="english", norm="l2")
    try:
        m = vec.fit_transform([a, b])
        return float(cosine_similarity(m[0:1], m[1:2])[0, 0])
    except ValueError:
        return 0.0


def compare(a, b):
    """ROUGE F1 (symmetric) + TF-IDF cosine for a pair of texts."""
    s = scorer.score(target=a or "", prediction=b or "")
    return {
        "rouge1": s["rouge1"].fmeasure,
        "rouge2": s["rouge2"].fmeasure,
        "rougeL": s["rougeL"].fmeasure,
        "cosine": cosine(a, b),
    }


print(run_path, "|", run["run_id"], "|", n, "models")

outputs\n8n\1f233aaa\judge_run.json | 1f233aaa | 5 models


## Judge Selection

`chosen[i]` — index of the critic whose refined version the judge selected for author `i`.

In [2]:
LABEL_RE = re.compile(r"LLM[-\s]?(\d+)")


def parse_selection(judgment, i):
    if not judgment or not judgment.strip():
        return None
    lines = [ln for ln in judgment.splitlines() if ln.strip()]
    for line in [ln for ln in lines if "select" in ln.lower()] + lines[:1] + [judgment]:
        m = LABEL_RE.search(line)
        if m:
            k = int(m.group(1)) - 1
            if 0 <= k < n and k != i and (refined[i][k] or "").strip():
                return k
            return None
    return None


chosen = [parse_selection(judgments[i], i) for i in range(n)]
for i in range(n):
    k = chosen[i]
    print(f"LLM-{i+1} {short[i]:<26} -> LLM-{k+1} {short[k]}" if k is not None
          else f"LLM-{i+1} {short[i]:<26} -> UNRESOLVED")

LLM-1 gpt-5.6-sol                -> LLM-5 claude-opus-4.8
LLM-2 kimi-k3                    -> LLM-1 gpt-5.6-sol
LLM-3 qwen3.7-max                -> LLM-5 claude-opus-4.8
LLM-4 gemini-3.1-pro-preview     -> LLM-1 gpt-5.6-sol
LLM-5 claude-opus-4.8            -> LLM-2 kimi-k3


## 1. Response Length — Primary Responses

In [3]:
t1 = pd.DataFrame([
    {"llm": f"LLM-{i+1}", "model": short[i], "words": words(primary[i])}
    for i in range(n)
])
t1

,llm,model,words
0,LLM-1,gpt-5.6-sol,2769
1,LLM-2,kimi-k3,3111
2,LLM-3,qwen3.7-max,1236
3,LLM-4,gemini-3.1-pro-preview,1046
4,LLM-5,claude-opus-4.8,996


## 2. Pairwise Similarity — Primary Responses

In [4]:
t2 = pd.DataFrame([
    {"pair": f"LLM-{i+1} vs LLM-{j+1}", "model_a": short[i], "model_b": short[j],
     **compare(primary[i], primary[j])}
    for i, j in combinations(range(n), 2)
])
t2.round(3)

,pair,model_a,model_b,rouge1,rouge2,rougeL,cosine
0,LLM-1 vs LLM-2,gpt-5.6-sol,kimi-k3,0.586,0.154,0.160,0.567
1,LLM-1 vs LLM-3,gpt-5.6-sol,qwen3.7-max,0.433,0.092,0.142,0.422
2,LLM-1 vs LLM-4,gpt-5.6-sol,gemini-3.1-pro-preview,0.427,0.121,0.146,0.532
3,LLM-1 vs LLM-5,gpt-5.6-sol,claude-opus-4.8,0.413,0.106,0.142,0.490
4,LLM-2 vs LLM-3,kimi-k3,qwen3.7-max,0.399,0.080,0.119,0.349
5,LLM-2 vs LLM-4,kimi-k3,gemini-3.1-pro-preview,0.376,0.092,0.126,0.411
6,LLM-2 vs LLM-5,kimi-k3,claude-opus-4.8,0.386,0.092,0.117,0.437
7,LLM-3 vs LLM-4,qwen3.7-max,gemini-3.1-pro-preview,0.621,0.208,0.245,0.562
8,LLM-3 vs LLM-5,qwen3.7-max,claude-opus-4.8,0.486,0.107,0.169,0.364
9,LLM-4 vs LLM-5,gemini-3.1-pro-preview,claude-opus-4.8,0.511,0.102,0.188,0.475


## 3. Response Length — Critiques

In [5]:
t3 = pd.DataFrame([
    {"target": f"LLM-{i+1}", "target_model": short[i],
     "critic": f"LLM-{j+1}", "critic_model": short[j],
     "words": words(critiques[i][j])}
    for i in range(n) for j in range(n)
    if i != j and critiques[i][j] is not None
])
t3

,target,target_model,critic,critic_model,words
0,LLM-1,gpt-5.6-sol,LLM-2,kimi-k3,1301
1,LLM-1,gpt-5.6-sol,LLM-3,qwen3.7-max,705
2,LLM-1,gpt-5.6-sol,LLM-4,gemini-3.1-pro-preview,694
3,LLM-1,gpt-5.6-sol,LLM-5,claude-opus-4.8,895
4,LLM-2,kimi-k3,LLM-1,gpt-5.6-sol,2210
5,LLM-2,kimi-k3,LLM-3,qwen3.7-max,616
6,LLM-2,kimi-k3,LLM-4,gemini-3.1-pro-preview,641
7,LLM-2,kimi-k3,LLM-5,claude-opus-4.8,1236
8,LLM-3,qwen3.7-max,LLM-1,gpt-5.6-sol,1990
9,LLM-3,qwen3.7-max,LLM-2,kimi-k3,1053


## 4. Pairwise Similarity — Critiques of the Same Primary Response

In [6]:
t4 = pd.DataFrame([
    {"target": f"LLM-{i+1}", "target_model": short[i],
     "pair": f"LLM-{j+1} vs LLM-{k+1}",
     **compare(critiques[i][j], critiques[i][k])}
    for i in range(n)
    for j, k in combinations([j for j in range(n) if j != i and critiques[i][j] is not None], 2)
])
t4.round(3)

,target,target_model,pair,rouge1,rouge2,rougeL,cosine
0,LLM-1,gpt-5.6-sol,LLM-2 vs LLM-3,0.440,0.085,0.136,0.337
1,LLM-1,gpt-5.6-sol,LLM-2 vs LLM-4,0.416,0.081,0.126,0.292
2,LLM-1,gpt-5.6-sol,LLM-2 vs LLM-5,0.581,0.132,0.161,0.452
3,LLM-1,gpt-5.6-sol,LLM-3 vs LLM-4,0.562,0.156,0.216,0.399
4,LLM-1,gpt-5.6-sol,LLM-3 vs LLM-5,0.455,0.088,0.153,0.316
5,LLM-1,gpt-5.6-sol,LLM-4 vs LLM-5,0.485,0.100,0.167,0.334
6,LLM-2,kimi-k3,LLM-1 vs LLM-3,0.314,0.066,0.102,0.248
7,LLM-2,kimi-k3,LLM-1 vs LLM-4,0.329,0.082,0.106,0.306
8,LLM-2,kimi-k3,LLM-1 vs LLM-5,0.491,0.095,0.125,0.335
9,LLM-2,kimi-k3,LLM-3 vs LLM-4,0.529,0.142,0.192,0.312


## 5. Response Length — Refined Responses

In [7]:
t5 = pd.DataFrame([
    {"author": f"LLM-{i+1}", "author_model": short[i],
     "critic": f"LLM-{j+1}", "critic_model": short[j],
     "words": words(refined[i][j])}
    for i in range(n) for j in range(n)
    if i != j and refined[i][j] is not None
])
t5

,author,author_model,critic,critic_model,words
0,LLM-1,gpt-5.6-sol,LLM-2,kimi-k3,4205
1,LLM-1,gpt-5.6-sol,LLM-3,qwen3.7-max,3234
2,LLM-1,gpt-5.6-sol,LLM-4,gemini-3.1-pro-preview,3268
3,LLM-1,gpt-5.6-sol,LLM-5,claude-opus-4.8,3595
4,LLM-2,kimi-k3,LLM-1,gpt-5.6-sol,4462
5,LLM-2,kimi-k3,LLM-3,qwen3.7-max,3482
6,LLM-2,kimi-k3,LLM-4,gemini-3.1-pro-preview,3742
7,LLM-2,kimi-k3,LLM-5,claude-opus-4.8,3374
8,LLM-3,qwen3.7-max,LLM-1,gpt-5.6-sol,1359
9,LLM-3,qwen3.7-max,LLM-2,kimi-k3,1486


## 6. Pairwise Similarity — Refined Responses of the Same Author

In [8]:
t6 = pd.DataFrame([
    {"author": f"LLM-{i+1}", "author_model": short[i],
     "pair": f"after LLM-{j+1} vs after LLM-{k+1}",
     **compare(refined[i][j], refined[i][k])}
    for i in range(n)
    for j, k in combinations([j for j in range(n) if j != i and refined[i][j] is not None], 2)
])
t6.round(3)

,author,author_model,pair,rouge1,rouge2,rougeL,cosine
0,LLM-1,gpt-5.6-sol,after LLM-2 vs after LLM-3,0.703,0.277,0.261,0.714
1,LLM-1,gpt-5.6-sol,after LLM-2 vs after LLM-4,0.671,0.251,0.231,0.686
2,LLM-1,gpt-5.6-sol,after LLM-2 vs after LLM-5,0.710,0.256,0.223,0.707
3,LLM-1,gpt-5.6-sol,after LLM-3 vs after LLM-4,0.739,0.326,0.334,0.782
4,LLM-1,gpt-5.6-sol,after LLM-3 vs after LLM-5,0.726,0.292,0.274,0.723
5,LLM-1,gpt-5.6-sol,after LLM-4 vs after LLM-5,0.696,0.274,0.269,0.711
6,LLM-2,kimi-k3,after LLM-1 vs after LLM-3,0.559,0.126,0.126,0.454
7,LLM-2,kimi-k3,after LLM-1 vs after LLM-4,0.607,0.163,0.142,0.545
8,LLM-2,kimi-k3,after LLM-1 vs after LLM-5,0.510,0.097,0.120,0.393
9,LLM-2,kimi-k3,after LLM-3 vs after LLM-4,0.662,0.208,0.197,0.589


## 7. Similarity — Primary vs. Refined Responses

In [9]:
t7 = pd.DataFrame([
    {"author": f"LLM-{i+1}", "author_model": short[i],
     "critic": f"LLM-{j+1}", "critic_model": short[j],
     "primary_words": words(primary[i]), "refined_words": words(refined[i][j]),
     **compare(primary[i], refined[i][j])}
    for i in range(n) for j in range(n)
    if i != j and refined[i][j] is not None
])
t7.round(3)

,author,author_model,critic,critic_model,primary_words,refined_words,rouge1,rouge2,rougeL,cosine
0,LLM-1,gpt-5.6-sol,LLM-2,kimi-k3,2769,4205,0.676,0.294,0.262,0.702
1,LLM-1,gpt-5.6-sol,LLM-3,qwen3.7-max,2769,3234,0.803,0.428,0.456,0.842
2,LLM-1,gpt-5.6-sol,LLM-4,gemini-3.1-pro-preview,2769,3268,0.758,0.342,0.342,0.833
3,LLM-1,gpt-5.6-sol,LLM-5,claude-opus-4.8,2769,3595,0.729,0.317,0.300,0.762
4,LLM-2,kimi-k3,LLM-1,gpt-5.6-sol,3111,4462,0.581,0.155,0.142,0.550
5,LLM-2,kimi-k3,LLM-3,qwen3.7-max,3111,3482,0.724,0.415,0.455,0.713
6,LLM-2,kimi-k3,LLM-4,gemini-3.1-pro-preview,3111,3742,0.696,0.265,0.240,0.689
7,LLM-2,kimi-k3,LLM-5,claude-opus-4.8,3111,3374,0.617,0.222,0.219,0.560
8,LLM-3,qwen3.7-max,LLM-1,gpt-5.6-sol,1236,1359,0.600,0.208,0.233,0.495
9,LLM-3,qwen3.7-max,LLM-2,kimi-k3,1236,1486,0.623,0.257,0.291,0.535


## 8. Response Length — Judge-Selected Responses

In [10]:
t8 = pd.DataFrame([
    {"author": f"LLM-{i+1}", "author_model": short[i],
     "chosen_critic": f"LLM-{chosen[i]+1}" if chosen[i] is not None else None,
     "chosen_critic_model": short[chosen[i]] if chosen[i] is not None else None,
     "words": words(refined[i][chosen[i]]) if chosen[i] is not None else None}
    for i in range(n)
])
t8

,author,author_model,chosen_critic,chosen_critic_model,words
0,LLM-1,gpt-5.6-sol,LLM-5,claude-opus-4.8,3595
1,LLM-2,kimi-k3,LLM-1,gpt-5.6-sol,4462
2,LLM-3,qwen3.7-max,LLM-5,claude-opus-4.8,1184
3,LLM-4,gemini-3.1-pro-preview,LLM-1,gpt-5.6-sol,1207
4,LLM-5,claude-opus-4.8,LLM-2,kimi-k3,2010


## 9. Pairwise Similarity — Judge-Selected Responses

In [11]:
chosen_text = [refined[i][chosen[i]] if chosen[i] is not None else None for i in range(n)]

t9 = pd.DataFrame([
    {"pair": f"LLM-{i+1} vs LLM-{j+1}", "model_a": short[i], "model_b": short[j],
     **compare(chosen_text[i], chosen_text[j])}
    for i, j in combinations(range(n), 2)
    if chosen_text[i] is not None and chosen_text[j] is not None
])
t9.round(3)

,pair,model_a,model_b,rouge1,rouge2,rougeL,cosine
0,LLM-1 vs LLM-2,gpt-5.6-sol,kimi-k3,0.600,0.148,0.133,0.559
1,LLM-1 vs LLM-3,gpt-5.6-sol,qwen3.7-max,0.350,0.072,0.113,0.319
2,LLM-1 vs LLM-4,gpt-5.6-sol,gemini-3.1-pro-preview,0.400,0.095,0.137,0.452
3,LLM-1 vs LLM-5,gpt-5.6-sol,claude-opus-4.8,0.457,0.089,0.117,0.331
4,LLM-2 vs LLM-3,kimi-k3,qwen3.7-max,0.301,0.062,0.100,0.270
5,LLM-2 vs LLM-4,kimi-k3,gemini-3.1-pro-preview,0.331,0.081,0.111,0.397
6,LLM-2 vs LLM-5,kimi-k3,claude-opus-4.8,0.413,0.076,0.106,0.320
7,LLM-3 vs LLM-4,qwen3.7-max,gemini-3.1-pro-preview,0.555,0.156,0.195,0.400
8,LLM-3 vs LLM-5,qwen3.7-max,claude-opus-4.8,0.463,0.082,0.133,0.257
9,LLM-4 vs LLM-5,gemini-3.1-pro-preview,claude-opus-4.8,0.457,0.077,0.136,0.298


## 10. Similarity — Primary vs. Judge-Selected Responses

In [12]:
t10 = pd.DataFrame([
    {"author": f"LLM-{i+1}", "author_model": short[i],
     "primary_words": words(primary[i]), "chosen_words": words(chosen_text[i]),
     **compare(primary[i], chosen_text[i])}
    for i in range(n) if chosen_text[i] is not None
])
t10.round(3)

,author,author_model,primary_words,chosen_words,rouge1,rouge2,rougeL,cosine
0,LLM-1,gpt-5.6-sol,2769,3595,0.729,0.317,0.300,0.762
1,LLM-2,kimi-k3,3111,4462,0.581,0.155,0.142,0.550
2,LLM-3,qwen3.7-max,1236,1184,0.727,0.451,0.488,0.703
3,LLM-4,gemini-3.1-pro-preview,1046,1207,0.593,0.242,0.273,0.561
4,LLM-5,claude-opus-4.8,996,2010,0.552,0.258,0.312,0.541


## Export

In [13]:
if SAVE_CSV:
    out = run_path.parent / "analysis"
    out.mkdir(exist_ok=True)
    tables = {
        "01_primary_length": t1,
        "02_primary_pairwise": t2,
        "03_critique_length": t3,
        "04_critique_pairwise": t4,
        "05_refined_length": t5,
        "06_refined_pairwise": t6,
        "07_primary_vs_refined": t7,
        "08_chosen_length": t8,
        "09_chosen_pairwise": t9,
        "10_primary_vs_chosen": t10,
    }
    for name, df in tables.items():
        df.to_csv(out / f"{name}.csv", index=False)
        print(f"{name:<24} {len(df):>3} rows")
    print("->", out)

01_primary_length          5 rows
02_primary_pairwise       10 rows
03_critique_length        20 rows
04_critique_pairwise      30 rows
05_refined_length         20 rows
06_refined_pairwise       30 rows
07_primary_vs_refined     20 rows
08_chosen_length           5 rows
09_chosen_pairwise        10 rows
10_primary_vs_chosen       5 rows
-> outputs\n8n\1f233aaa\analysis
